Here is a **comprehensive report** addressing all required components of the assignment. The report is structured clearly for submission. Since I cannot run live code or access a personal photo of your face here, I provide complete, ready-to-use **PyTorch code snippets** (the most common framework for such tasks) that you can execute in a Jupyter notebook or Colab with a GPU. Replace the face image path with your own photo.

### 1. Visualization and Analysis of Feature Maps from Different CNN Layers (Using Your Own Face Image)

**Selected CNN**: Pre-trained **VGG16** (from torchvision), a classic CNN with clear hierarchical feature extraction. Early layers detect edges/textures; deeper layers capture complex patterns like faces or objects.

**Steps**:
- Load your face image (e.g., a selfie, resized to 224×224 for VGG16).
- Preprocess it (normalize with ImageNet stats).
- Hook into convolutional layers to extract feature maps.
- Visualize selected channels (e.g., first 64 or summed/averaged for clarity).

**Code Example (PyTorch)**:

```python
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Load pre-trained VGG16
model = models.vgg16(pretrained=True)
model.eval()

# Extract conv layers
conv_layers = []
for layer in model.features:
    if isinstance(layer, nn.Conv2d):
        conv_layers.append(layer)

print(f"Total conv layers: {len(conv_layers)}")

# Preprocess your face image (replace with your photo path)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img = Image.open("your_face_photo.jpg").convert("RGB")
input_img = transform(img).unsqueeze(0)  # Add batch dim

# Hook to get feature maps
feature_maps = []
layer_names = []
x = input_img
for i, layer in enumerate(conv_layers):
    x = layer(x)
    feature_maps.append(x.detach())
    layer_names.append(f"Conv Layer {i+1}")

# Visualize feature maps (example: first 8 channels of selected layers)
def visualize_feature_maps(feature_map, layer_idx, num_channels=8):
    fm = feature_map[0]  # Remove batch
    fig, axs = plt.subplots(1, num_channels, figsize=(20, 5))
    for i in range(num_channels):
        axs[i].imshow(fm[i].cpu().numpy(), cmap='viridis')
        axs[i].axis('off')
    plt.suptitle(f"Feature Maps - {layer_names[layer_idx]}")
    plt.show()

# Example visualizations
visualize_feature_maps(feature_maps[0], 0)   # Early layer: edges, textures
visualize_feature_maps(feature_maps[5], 5)   # Mid layer: patterns, parts of face
visualize_feature_maps(feature_maps[12], 12) # Deeper layer: higher-level features
```

**Analysis**:
- **Early layers** (e.g., Conv1–Conv3): Feature maps show low-level features like edges, corners, and basic textures of your face (e.g., hair lines, eye contours). Activations are sparse and localized.
- **Mid layers**: More complex patterns emerge—facial structures, shadows, or skin textures. Receptive fields grow, capturing combinations of edges.
- **Deeper layers**: Highly abstract; maps highlight semantic regions (e.g., eyes, mouth as "face parts"). Many channels may activate strongly on your facial landmarks, demonstrating hierarchical learning.
- Insight: As depth increases, spatial resolution decreases slightly (due to pooling in VGG), but semantic richness increases. This explains why CNNs excel at object/face recognition—early layers are general, later ones task-specific.

Run this with your photo to observe how your face is "dissected" layer by layer. You can average channels or use Grad-CAM for class-specific heatmaps if extending the analysis.

### 2. Design and Architectural Justification of a Custom CNN Model

**Proposed Custom CNN** (for CIFAR-10, 32×32×3 images, 10 classes):

I design a moderately deep CNN inspired by modern practices but kept simple for training efficiency on CIFAR-10.

**Architecture** (`CustomCIFARNet`):

```python
import torch.nn as nn
import torch.nn.functional as F

class CustomCIFARNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Block 1: Early feature extraction
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool1 = nn.MaxPool2d(2, 2)  # 16x16
        
        # Block 2: Deeper features
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool2 = nn.MaxPool2d(2, 2)  # 8x8
        
        # Block 3: Further abstraction
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        self.pool3 = nn.MaxPool2d(2, 2)  # 4x4
        
        # Classifier
        self.fc1 = nn.Linear(512 * 4 * 4, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 10)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        
        x = x.view(-1, 512 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x
```

**Architectural Justification**:
- **Convolutional blocks with increasing channels** (64 → 128 → 256 → 512): Allows hierarchical feature learning—more filters capture richer representations deeper in the network.
- **3×3 kernels with padding=1**: Standard for preserving spatial dimensions before pooling; efficient and effective (as in VGG/ResNet).
- **Batch Normalization** after each conv: Stabilizes training, reduces internal covariate shift, allows higher learning rates, and acts as mild regularization.
- **MaxPooling (2×2)**: Reduces spatial dimensions (from 32→16→8→4), lowers computation, and provides translation invariance.
- **Dropout (0.5) before final FC**: Prevents overfitting on CIFAR-10's relatively small dataset.
- **No global average pooling** here (kept simple FC head); could be added for fewer parameters.
- Overall: Balanced depth/width for CIFAR-10 (aims for ~80-85%+ accuracy with proper training). It builds on proven designs while remaining trainable quickly.

This model has a good parameter-accuracy trade-off and is easy to modify (e.g., add residual connections for deeper variants).

### 3. Implementation and Training of the Proposed CNN for Multi-Class Classification on CIFAR-10

**CIFAR-10 Overview**: 60,000 32×32 color images in 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). 50,000 train, 10,000 test.

**Full Training Code** (PyTorch; use DataLoader with augmentations for better results):

```python
import torch
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data with augmentation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
transform_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# Model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = CustomCIFARNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

# Training loop (e.g., 50 epochs)
for epoch in range(50):
    net.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}")

# Evaluation
net.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Test Accuracy: {100 * correct / total:.2f}%")
```

**Expected Results**: With this setup and ~50 epochs, you can achieve **~80-85% test accuracy** (higher with more epochs, ResNet-like additions, or better optimizers like SGD with momentum). Monitor training/validation curves for overfitting.

### 4. Comparative Analysis of the Effects of Three Different Activation Functions

**Chosen Activations**: **ReLU**, **Leaky ReLU** (negative slope=0.01), and **Swish** (or SiLU: x * sigmoid(x)).

**Implementation**: Modify the model by replacing `F.relu` with the chosen activation (or use `nn.LeakyReLU()`, `nn.SiLU()`). Train three identical models separately on CIFAR-10 (same hyperparameters, seeds for fairness). Track:
- Training/validation accuracy & loss curves.
- Convergence speed.
- Final test accuracy.
- Dying neuron issues (for ReLU).

**Key Findings from Literature/Experiments**:
- **ReLU**: Fastest training, simple, promotes sparsity. Good baseline (~80%+ on CIFAR-10). Can suffer from "dying ReLU" (neurons stuck at 0).
- **Leaky ReLU**: Addresses dying ReLU by allowing small negative gradients. Often similar or slightly better accuracy; helps in deeper networks but may train slightly slower.
- **Swish**: Smoother, non-monotonic. Frequently outperforms ReLU on CIFAR-10 (1-2% gains in some studies), better gradient flow, but computationally heavier (sigmoid involved). Can lead to better generalization and faster convergence in mid-training.

**Comparative Summary** (typical observed trends):
- **Accuracy**: Swish ≥ Leaky ReLU > ReLU (small margins on CIFAR-10).
- **Learning Dynamics**: Swish shows smoother loss curves and less sensitivity to initialization. ReLU converges quickest per epoch but may plateau.
- **Training Time**: ReLU fastest; Swish slowest due to extra ops.
- Recommendation: Use Swish or Leaky ReLU for modern models; ReLU for simplicity/speed.

Plot loss/accuracy curves side-by-side for visual comparison in your report.

### 5. Critical Discussion on the Impact of Various Convolutional Kernel Types

**1. Regular (Standard) Kernels** (e.g., 3×3 Conv2d):
- Full channel mixing + spatial filtering in one step.
- High parameter/compute cost: For input channels M, output N, kernel K×K → ~K²×M×N parameters/ops.
- Strong baseline performance but inefficient for mobile/edge devices.

**2. Deformable Kernels**:
- Learnable 2D offsets added to sampling grid → kernel "deforms" adaptively to input geometry.
- Impact: Excellent for irregular shapes, deformations, or non-rigid objects (e.g., better object detection/segmentation). Improves accuracy on varied datasets without fixed grid limitations. Adds minor overhead (offset prediction conv). Useful in your custom model for complex CIFAR objects.

**3. Dilated (Atrous) Kernels**:
- Introduces dilation rate r > 1 (inserts gaps between kernel weights).
- Impact: Expands receptive field exponentially without losing resolution or adding parameters (e.g., rate=2 makes 3×3 cover 5×5 area). Great for multi-scale context (segmentation, Dense Prediction). Risk of "gridding" artifact if rates not chosen carefully. Complements pooling in your model for broader context on 32×32 images.

**4. Depthwise Separable Kernels** (Depthwise + Pointwise):
- Depthwise:  K×K filter per input channel (spatial only).
- Pointwise: 1×1 conv to mix channels.
- Impact: Dramatically reduces parameters/compute (~8-9× fewer ops vs. regular for 3×3). Core of MobileNet—enables efficient models with near-comparable accuracy. Ideal for resource-constrained training/inference. Slightly lower accuracy than regular but excellent efficiency-accuracy trade-off.

**5. Modified Depthwise-Separable Kernels**:
- Variants (e.g., with added residuals, attention, or grouped variants). Often improve upon basic depthwise by addressing limitations like limited cross-channel info early on. Can boost accuracy while retaining efficiency.

**6. Pointwise Kernels** (1×1 Conv):
- Pure channel mixing (no spatial).
- Impact: Used after depthwise for dimension reduction/expansion. Extremely cheap; reduces channels before expensive ops. Common in bottlenecks (e.g., Inception, MobileNet). Enhances efficiency when combined with others.

**Critical Comparison**:
- **Efficiency**: Depthwise separable + pointwise >> regular > dilated/deformable (latter add flexibility at moderate cost).
- **Performance**: Regular/deformable often highest accuracy; dilated for context-heavy tasks; separable for lightweight deployment.
- **Use Cases in Your Model**: Replace some regular convs with depthwise separable for faster training/smaller size. Add dilated in deeper layers for larger receptive fields on CIFAR. Deformable shines if extending to detection/segmentation.
- Trade-offs: Flexibility (deformable/dilated) vs. efficiency (separable) vs. simplicity (regular). Hybrid designs (e.g., MixConv or MobileNet-style) often win in practice.

